# Training a Classifier on CIFAR10 Dataset

In this notebook, we will use the CIFAR10 dataset, which has ten classes: `airplane`, `automobile`, `bird`, `cat`, `deer`, `dog`, `frog`, `horse`, `ship`, `truck`. The images' size i 3x32x32, i.e., 3-channel color images of 32x32 pixels in size.

For training the model, we will do the following steps in order:

1. Load and normalize the CIFAR10 training and test sets using `torchvision`
2. Define a convolutional neural network
3. Define a loss function
4. Train the network on the training set
5. Evaluate the network on the test set


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

import torch
import torchvision
from   torchvision         import transforms
import torch.nn            as     nn
import torch.nn.functional as     F
import torch.optim         as     optim

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## 1. Load and normalize the CIFAR10 images

The output of a `torchvision` dataset are PILImage images eith values in the [0, 1] range. So, we transform them to Tensors and normalize the values to the [-1, 1]  range.


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(
    root      = './data',
    train     = True,
    download  = True,
    transform = transform,
)

trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size  = 4,
    shuffle     = True,
    num_workers = 4,
)

testset = torchvision.datasets.CIFAR10(
    root      = './data',
    train     = False,
    download  = True,
    transform = transform,
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size  = 4,
    shuffle     = False,
    num_workers = 4,
)

classes = (
    'plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck'
)

Let us visualize some of the training images.

In [ ]:
# Function to plot an image.

def imshow(img):
    img   = img / 2 + 0.5  # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# Get some random training images
batch          = iter(trainloader)
images, labels = next(batch)

# Display the loaded images
grid = torchvision.utils.make_grid(images)
imshow(grid)

# Print the image labels
print(' '.join('%5s' % classes[labels[j]] for j in range(4)))


## 2. Define a convolutional neural network

Define a neural network with a Conv2d-Maxpool-Conv2d section, and a Linear-ReLU pair, a second Linear-ReLU pair, and a final Linear layer.


In [ ]:
class CNN(nn.Module):

    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool  = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1   = nn.Linear(16 * 5 * 5, 120)
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


In [ ]:
model = CNN().to(device)

## 3. Define the loss function and select the optimizer

We will use the Cross-Entropy loss and SGD with momentum as optimizer.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)


## 4. Train the CNN model

We simply have to iterte over our training data loader, and feed the loaded images into the model input and then run the backward pass to optimize the model.

In [ ]:
# Iterate over the dataset multiple times

epochs = 5

for epoch in range(epochs:  

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):

        # Get a batch of images and labels (as a list of two tensors)
        images, labels = data[0].to(device), data[1].to(device)

        # Reset the gradients
        optimizer.zero_grad()

        # forward pass
        predictions = model(images)
        
        # Calculate the loss
        loss = criterion(predictions, labels)
        
        # Calculate the loss gradients
        loss.backward()

        # Update the model weights
        optimizer.step()

        # Print training statistics every 2000 iterations/batches
        running_loss += loss.item()
        if i % 2000 == 1999:  
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 2000))
            running_loss = 0.0

print('[INFO] Finished training')

## Save the trained model to disk

In [ ]:
PATH = './results/cnn_cifar10_dense.pth'

os.make_dirs('./results', exist_ok=True)

torch.save(model.state_dict(), PATH)

## 5. Evaluate the model on the test set

Verify if the network has been trained effectively.

We will check this by predicting the class of each input image and compare the prediction with the ground-truth label. If the prediction is correct, we update the number of correct predictions.

Let us first display an image from the test set.

In [ ]:
data_iterator  = iter(testloader)
images, labels = next(data_iterator)

# Display the images
grid = torchvision.utils.make_grid(images)
imshow(grid)

# Print the labels
print('Classes: ', ' '.join('%5s' % classes[labels[j]] for j in range(4)))

Next, let us load the saved model, although in a normal execution of the notebook, we would not need to load the saved model.

In [ ]:
model = CNN()
model.load_state_dict(torch.load(PATH))

Let us predict the labels, of the images loaded from the test set, with our model.

In [ ]:
outputs = model(images)

The outputs are the logits for the 10 classes. So, let us get the index of the highest logit.

In [ ]:
_, predictions = torch.max(outputs, 1)

print('Predicted: ', ' '.join('%5s' % classes[predictions[j]] for j in range(4)))

Let us see how the models performs on the whole test set.

In [ ]:
correct = 0
total   = 0

with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs        = model(images)
        _, predictions = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predictions == labels).sum().item()

print(f'Model accuracy on the 10000 test images: {(100 * correct) / total :.1f}%')


Let us identify the classes that performed the best and the worst.

In [ ]:
class_correct = list(0. for i in range(10))
class_total    = list(0. for i in range(10))
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs        = model(images)
        _, predicted   = torch.max(outputs, 1)
        c = (predicted == labels).squeeze()
        for i in range(4):
            label                 = labels[i]
            class_correct[label] += c[i].item()
            class_total[label]   += 1

for i in range(10):
    print('Accuracy of %5s : %2d %%' %
          (classes[i], 100 * class_correct[i] / class_total[i]))
